In [ ]:
import json
import pandas as pd
import requests
import numpy as np

In [ ]:
# 엑셀 파일에서 데이터 읽기
demand_df = pd.read_excel('/content/거리행렬_수요지(1~20).xlsx')  # 수요지 데이터
candidates_df = pd.read_excel('/content/후보지.xlsx')  # 후보지 데이터

In [ ]:
# 수요지와 후보지의 위도 경도 정보 추출
demand_locations = demand_df[['위도', '경도']].values
candidate_locations = candidates_df[['위도', '경도']].values

In [ ]:
# TMAP API 키
app_key = ""  # 여기에 발급받은 TMAP API 키를 입력하세요.

# 거리 행렬 초기화
distance_matrix = np.zeros((len(demand_locations), len(candidate_locations)))

# 각 수요지와 후보지 쌍에 대해 API 요청
for i, demand_loc in enumerate(demand_locations):
    for j, candidate_loc in enumerate(candidate_locations):

        # 요청 데이터
        url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1&format=json"

        payload = {
            "startX": str(demand_loc[1]),
            "startY": str(demand_loc[0]),
            "endX": str(candidate_loc[1]),
            "endY": str(candidate_loc[0]),
            "reqCoordType": "WGS84GEO",
            "resCoordType": "WGS84GEO",
            "startName": "출발지",
            "endName": "도착지"
        }

        headers = {
            "accept": "application/json",
            "content-type": "application/json",
            "appKey": app_key
        }

        # API 요청
        response = requests.post(url, json=payload, headers=headers)

        # 응답 처리
        if response.status_code == 200:
            result_data = response.json().get('features', [])
            if result_data:
                total_distance = result_data[0]['properties']['totalDistance']  # 단위는 미터
                distance_matrix[i, j] = total_distance  # m 단위로 변환하여 저장
                print("거리 저장")
            else:
                print("경로 정보를 찾을 수 없음")
                distance_matrix[i, j] = np.inf  # 경로 정보를 찾을 수 없는 경우 무한대로 설정
        else:
            print("API 요청 실패")
            distance_matrix[i, j] = np.inf  # API 요청 실패 시 무한대로 설정

# 거리 행렬 출력
print("거리 행렬 (km):")
print(distance_matrix)


In [ ]:
pd.DataFrame(distance_matrix).to_excel('거리행렬_수요지후보지(1~20).xlsx', index=False)